<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# AfriCare Support Analytics
## Notebook 3 - SQL Analytics, KPIs & Performance
### VERSION CORRIGEE - Mode formateur

---

> **Comment lire ce notebook :**
> Chaque bloc SQL est precede d'une cellule qui explique la logique de la requete,
> les patterns SQL utilises et l'insight metier attendu.
> Les blocs `POINT PEDAGOGIQUE` expliquent les regles generalisables a tout projet data.
> Les blocs `INSIGHT` contiennent la lecture des resultats pour M. Kouame.

---

| Info | Detail |
|---|---|
| **Prerequis** | Notebook 2 complete - `support_clean_analytics.csv` genere |
| **Objectif** | Repondre aux questions business avec SQL, analyser SLA, agents, backlog, CSAT |
| **Sortie attendue** | KPIs chiffres, graphiques de performance, synthese actionnable |
| **Duree estimee** | 4h a 5h |
| **Competences SQL** | Agregations - CASE WHEN - Window functions (RANK, LAG) - CTE - JOIN |

---

### Plan du notebook

```
1. Connexion DuckDB & chargement
2. KPIs globaux operationnels
3. Performance SLA par categorie
4. Performance des agents (RANK, heatmap normalisee)
5. Evolution temporelle (LAG, heatmap heure x jour)
6. Angles morts du systeme d'alerte SLA
7. Satisfaction client CSAT
8. Synthese des insights
```

---

# 1. Imports & connexion DuckDB

> **POINT PEDAGOGIQUE - Pourquoi DuckDB plutot que SQL Server dans ce notebook ?**
>
> SQL Server est l'environnement de production recommande (cf. Notebook 2).
> DuckDB est une alternative **portable** qui ne necessite aucune installation serveur.
> Il lit directement les CSV avec `read_csv_auto` et supporte une syntaxe quasi-identique
> a PostgreSQL et SQL Server.
>
> **Correspondances de syntaxe a retenir :**
> - `DATE_TRUNC('month', date)` -> DuckDB / PostgreSQL
> - `DATEADD(month, DATEDIFF(month,0,date), 0)` -> SQL Server equivalent
> - `HOUR(timestamp)` -> DuckDB ; `DATEPART(hour, date)` -> SQL Server
>
> Les requetes SQL de ce notebook fonctionnent dans SQL Server avec ces ajustements mineurs.
>
> **La palette `COLORS` :**
> On redefinit ici la meme palette semantique que dans les notebooks precedents.
> Rouge = danger, orange = vigilance, vert = bon, violet = principal.
> Cette coherence visuelle est une bonne pratique professionnelle :
> les memes couleurs dans Python, Power BI et les slides.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os, sys, warnings
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/customer_support_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}")

# ── Chemins des fichiers ────────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/data/"
clean_path = f"{SAVE_PATH}support_clean_analytics.csv"




In [ ]:
# pour executer du code SQL dans le notebook
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
# ── Connexion DuckDB ────────────────────────────────────────────────────────
# tickets  → fichier nettoyé produit par le Notebook 2 (SAVE_PATH)
# autres   → fichiers bruts depuis GitHub
import duckdb

# Connexion DuckDB
conn = duckdb.connect()
conn.execute(f"""
    CREATE OR REPLACE  TABLE tickets      AS SELECT * FROM read_csv_auto('{BASE_URL}tickets.csv');
    CREATE OR REPLACE  TABLE agents       AS SELECT * FROM read_csv_auto('{BASE_URL}agents.csv');
    CREATE OR REPLACE  TABLE categories   AS SELECT * FROM read_csv_auto('{BASE_URL}categories.csv');
    CREATE OR REPLACE  TABLE interactions AS SELECT * FROM read_csv_auto('{BASE_URL}interactions.csv');
    CREATE OR REPLACE  TABLE sla_alerts   AS SELECT * FROM read_csv_auto('{BASE_URL}sla_alerts.csv');
""")

print(f"✅ {conn.execute('SELECT COUNT(*) FROM tickets').fetchone()[0]:,} tickets chargés")
print("Tables disponibles : tickets, agents, categories, interactions, sla_alerts")


In [ ]:
# Lier la connexion Python à JupySQL pour les cellules %%sql
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

---

# 2. KPIs globaux operationnels

> **POINT PEDAGOGIQUE - La requete de tableau de bord : tout calculer en une passe.**
>
> En production, les dashboards Power BI rafraichissent leurs KPIs souvent.
> Une requete qui calcule tous les indicateurs en un seul SELECT est bien plus performante
> que 10 requetes separees : une seule lecture de la table, un seul passage sur les donnees.
>
> **Le pattern fondamental pour calculer un taux en SQL :**
>
> ```sql
> AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END) * 100
> ```
>
> Decortiquons-le etape par etape :
> - `CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END` transforme chaque ligne en 0 ou 1
> - `AVG(...)` fait la moyenne de ces 0 et 1 = proportion de lignes ou la condition est vraie
> - `* 100` convertit la proportion en pourcentage
>
> C'est l'equivalent SQL de `df['sla_breach'].mean() * 100` en pandas.
> Ce pattern fonctionne dans **tous** les dialectes SQL sans exception.
>
> **`ROUND(..., 1)` systematique :**
> Toujours arrondir les resultats affiches. Un flottant brut comme `47.3182741...`
> est illisible dans un rapport. La precision au-dela d'une decimale n'apporte aucune
> valeur pour un KPI operationnel.

In [ ]:
%%sql df_kpi <<
SELECT
        COUNT(*)                                                              AS nb_tickets_total,
        SUM(CASE WHEN statut IN ('Resolu','Ferme') THEN 1 ELSE 0 END)        AS nb_resolus,
        ROUND(AVG(CASE WHEN statut IN ('Resolu','Ferme') THEN 1.0 ELSE 0.0 END)*100, 1)
                                                                              AS taux_resolution_pct,
        ROUND(AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END)*100, 1)     AS taux_sla_breach_pct,
        ROUND(AVG(CASE WHEN in_backlog=1 THEN 1.0 ELSE 0.0 END)*100, 1)     AS taux_backlog_pct,
        ROUND(AVG(CASE WHEN statut='Escalade' THEN 1.0 ELSE 0.0 END)*100, 1) AS taux_escalade_pct,
        ROUND(AVG(first_response_heures), 2)                                  AS first_response_moy_h,
        ROUND(AVG(resolution_heures), 2)                                      AS resolution_moy_h,
        ROUND(AVG(csat), 2)                                                   AS csat_moyen,
        SUM(reopened)                                                          AS nb_rouverts,
        ROUND(AVG(CAST(reopened AS FLOAT))*100, 1)                           AS taux_reouverture_pct
    FROM tickets


In [ ]:
cibles = {
    "taux_sla_breach_pct": ("< 10%",  "CRITIQUE"),
    "taux_backlog_pct":    ("< 3%",   "ELEVE"),
    "taux_escalade_pct":   ("< 5%",   "A surveiller"),
    "csat_moyen":          ("> 4.5",  "Sous objectif"),
}

print("=" * 60)
print("  KPIs GLOBAUX - AFRICAIRE SUPPORT")
print("=" * 60)
for col in df_kpi.columns:
    val = df_kpi[col].iloc[0]
    if col in cibles:
        c, a = cibles[col]
        print(f"  {col:<35}: {val}  [cible : {c}]  --> {a}")
    else:
        print(f"  {col:<35}: {val}")

### Lecture des KPIs

> **POINT PEDAGOGIQUE - Un tableau de KPIs sans lecture n'est pas un livrable.**
>
> L'analyste doit toujours produire une lecture en langage metier.
> Ce paragraphe est ce qu'on envoie a M. Kouame - pas le tableau brut.
>
> **Format recommande :** chiffre precis -> comparaison a la cible -> implication -> recommandation.

In [ ]:
kpi = df_kpi.iloc[0]

lines = [
    "=" * 65,
    "  LECTURE DES KPIs POUR M. KOUAME",
    "=" * 65,
    "",
    f"Volume : {kpi['nb_tickets_total']:,.0f} tickets. Taux de resolution : {kpi['taux_resolution_pct']:.1f}%",
    "",
    f"SLA : {kpi['taux_sla_breach_pct']:.1f}% de breach - quasi 1 ticket sur 2.",
    f"  Objectif standard du secteur : < 10%. Ecart : {kpi['taux_sla_breach_pct']-10:.0f} points.",
    "",
    f"Backlog : {kpi['taux_backlog_pct']:.1f}% de tickets bloques. Objectif < 3%.",
    "",
    f"Premiere reponse moyenne : {kpi['first_response_moy_h']:.1f}h. Resolution moyenne : {kpi['resolution_moy_h']:.1f}h.",
    "",
    f"CSAT moyen : {kpi['csat_moyen']:.2f}/5. Un CSAT < 4.0 est associe a un risque de churn eleve.",
    "",
    f"Tickets rouverts : {kpi['taux_reouverture_pct']:.1f}% - chaque reouverture = insatisfaction client",
    "  + surcout operationnel pour l'equipe.",
]
print("\n".join(lines))

---

# 3. Performance SLA par categorie

> **POINT PEDAGOGIQUE - Le JOIN et ses implications.**
>
> ```sql
> FROM tickets t
> JOIN categories c ON t.category_id = c.category_id
> ```
>
> Le JOIN (inner join) ne garde que les lignes avec une correspondance dans les deux tables.
> Les alias `t.` et `c.` evitent l'ambiguite sur les colonnes communes.
> Ici `sla_heures` existe dans les deux tables : sans alias, DuckDB leverait une erreur.
>
> **La colonne `depassement_moy_h` - la plus informative de cette requete :**
>
> ```sql
> AVG(CASE WHEN t.sla_breach=1 THEN t.resolution_heures - t.sla_heures END)
> ```
>
> Ce calcul ne fait la moyenne que sur les tickets en breach.
> Pour les tickets non en breach, la condition renvoie NULL et AVG ignore les NULLs.
> Resultat : combien d'heures AU-DELA du SLA les tickets tardifs ont-ils attendu.
>
> Un taux de 50% avec un depassement moyen de 2h est bien moins grave qu'un taux
> de 50% avec 48h de depassement. Le taux seul est insuffisant pour piloter.

In [ ]:
%%sql df_sla_cat <<
SELECT
        c.nom,
        c.domaine,
        c.sla_heures,
        c.sla_strict,
        COUNT(*)                                                                       AS nb_tickets,
        ROUND(AVG(CASE WHEN t.sla_breach=1 THEN 1.0 ELSE 0.0 END)*100, 1)            AS taux_breach,
        ROUND(AVG(t.resolution_heures), 1)                                             AS resolution_moy_h,
        ROUND(AVG(t.csat), 2)                                                          AS csat_moy,
        ROUND(AVG(t.ratio_sla), 3)                                                     AS ratio_sla_moy,
        ROUND(AVG(CASE WHEN t.sla_breach=1
                       THEN t.resolution_heures - t.sla_heures END), 1)               AS depassement_moy_h
    FROM tickets t
    JOIN categories c ON t.category_id = c.category_id
    GROUP BY c.nom, c.domaine, c.sla_heures, c.sla_strict
    ORDER BY taux_breach DESC


In [ ]:
print("Performance SLA par categorie (triee par taux de breach decroissant) :")
display(df_sla_cat)

cats_critiques = df_sla_cat[df_sla_cat["taux_breach"] > 50]
print(f"\nCategories avec breach > 50% : {len(cats_critiques)}")
for _, r in cats_critiques.iterrows():
    print(f"  {r['nom']} : {r['taux_breach']}% breach - depassement moy : {r['depassement_moy_h']}h")

Les résultats sont frappants.

**Demande info — 68.2% de breach.** Le pire score.
Et c'est une catégorie à SLA non-strict — mais 68% reste inacceptable.

**Facturation — 63% de breach.** Et celle-là, c'est `sla_strict = True`.
Des engagements contractuels violés sur 63% des tickets de facturation.

**Livraison retardée et Remboursement — ~51-52%.**
Deux catégories Billing avec SLA strict au-dessus de 50%.

À l'autre extrémité — **Panne technique et Problème paiement : 38%.**
Ces catégories ont un SLA de 120 heures — plus court, mais mieux respecté.

Le paradoxe : les catégories avec les SLA les plus courts performent mieux.
Pourquoi ? Parce que les équipes techniques traitent ces tickets en priorité.
Les tickets "Demande info" — perçus comme moins urgents — sont relégués au fond de la file.

### Visualisation - Deux angles complementaires

> **POINT PEDAGOGIQUE - Deux graphiques, deux questions differentes.**
>
> **Graphique gauche - barres horizontales :**
> Repond a "Quelles categories ont le plus de breach ?"
> Couleur semantique : rouge > 50%, orange > 35%, vert sinon.
> La ligne a 50% materialise la frontiere critique.
>
> **Graphique droit - scatter plot 3 dimensions :**
> Repond a "Y a-t-il un lien entre SLA contractuel et taux de breach ?"
> Axe X = duree SLA contractuel, Axe Y = taux breach observe, Taille = volume tickets.
>
> Si les categories avec des SLA courts breachent plus, les SLA sont trop ambitieux.
> Cette info justifie une renegociation contractuelle.
>
> `s = nb_tickets * 0.5 + 50` : la taille des points en matplotlib est en points^2 (surface).
> Le `+ 50` garantit une taille minimale visible pour les petits volumes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_b = [COLORS["danger"]  if v > 50 else
            COLORS["warning"] if v > 35 else
            COLORS["secondary"] for v in df_sla_cat["taux_breach"]]

axes[0].barh(df_sla_cat["nom"], df_sla_cat["taux_breach"],
              color=colors_b, alpha=0.85, edgecolor="white")
axes[0].axvline(50, color=COLORS["danger"],    linestyle="--", linewidth=1.5, alpha=0.7, label="Seuil 50%")
axes[0].axvline(10, color=COLORS["secondary"], linestyle="--", linewidth=1.5, alpha=0.7, label="Objectif < 10%")
axes[0].set_xlabel("Taux de breach SLA (%)")
axes[0].set_title("Taux de breach par categorie", fontweight="bold")
axes[0].legend(fontsize=9)

sc = axes[1].scatter(
    df_sla_cat["sla_heures"], df_sla_cat["taux_breach"],
    s=df_sla_cat["nb_tickets"] * 0.5 + 50,
    c=df_sla_cat["taux_breach"], cmap="RdYlGn_r", vmin=0, vmax=80,
    alpha=0.85, edgecolors="white", linewidths=1.5
)
for _, r in df_sla_cat.iterrows():
    axes[1].annotate(r["nom"].split()[0], (r["sla_heures"], r["taux_breach"]),
                     xytext=(5, 4), textcoords="offset points", fontsize=8)
plt.colorbar(sc, ax=axes[1], label="Taux breach (%)", shrink=0.8)
axes[1].set_xlabel("SLA contractuel (heures)")
axes[1].set_ylabel("Taux de breach (%)")
axes[1].set_title("SLA contractuel vs Taux breach\n(taille = volume tickets)", fontweight="bold")

plt.tight_layout()
plt.savefig("sla_categories.png", dpi=150, bbox_inches="tight")
plt.show()

corr = df_sla_cat["sla_heures"].corr(df_sla_cat["taux_breach"])
print(f"Correlation SLA contractuel / Taux breach : {corr:.2f}")
print("  Si proche de -1 : les SLA courts sont plus souvent breaches (SLA trop ambitieux)")
print("  Si proche de 0  : le SLA contractuel n'explique pas le breach")

**Le premier graphique** montre les taux de breach par catégorie.

La ligne pointillée rouge à 50% est le seuil d'alerte.
Tout ce qui dépasse cette ligne est une urgence opérationnelle.

**Le deuxième graphique** est le plus révélateur.

Axe X : le SLA contractuel en heures.
Axe Y : le taux de breach.
La taille des bulles : le volume de tickets.

On voit clairement deux groupes.

Les catégories avec des SLA courts (120-180 heures) — Panne technique, Problème paiement —
ont les taux de breach les plus bas. Elles sont traitées avec urgence.

Les catégories avec des SLA longs (360-480 heures) — Demande info, Qualité produit —
ont les taux de breach les plus élevés. Paradoxalement, plus on donne de temps, plus on le gaspille.

C'est la **loi de Parkinson** appliquée au support client :
le travail s'étend pour remplir le temps disponible.

---

# 4. Performance des agents - Window Functions

> **POINT PEDAGOGIQUE - RANK() OVER : la window function de classement.**
>
> Une window function calcule une valeur pour chaque ligne en regardant un ensemble
> de lignes defini par OVER(...) - sans reduire le nombre de lignes comme GROUP BY.
>
> ```sql
> RANK() OVER (ORDER BY AVG(sla_breach) ASC)          AS rank_sla
> RANK() OVER (ORDER BY AVG(csat) DESC NULLS LAST)    AS rank_csat
> ```
>
> **RANK() vs ROW_NUMBER() vs DENSE_RANK() :**
> - `ROW_NUMBER()` : rangs uniques meme si egalite (1, 2, 3, 4...)
> - `RANK()` : meme rang si egalite, saute le suivant (1, 1, 3...)
> - `DENSE_RANK()` : meme rang si egalite, ne saute pas (1, 1, 2...)
>
> Pour un classement de performance, RANK() est le plus juste : deux agents
> avec exactement le meme score meritent le meme rang.
>
> **`NULLS LAST` dans ORDER BY :**
> Sans cette option, les NULLs apparaissent en premier dans un tri descendant.
> Un agent sans note CSAT se retrouverait en "meilleur" rang - absurde.
> NULLS LAST force les NULLs a la fin, quelle que soit la direction du tri.
>
> **`ecart_csat = AVG(csat_reel) - csat_historique` :**
> Mesure si l'agent s'ameliore (positif) ou regresse (negatif) par rapport
> a sa performance historique. C'est une metrique de **progression**, pas absolue.

In [ ]:
%%sql df_agt <<
SELECT
        a.nom,
        a.tier,
        a.bureau,
        a.csat_moyen                                                                AS csat_hist,
        COUNT(*)                                                                     AS nb_tickets,
        ROUND(AVG(CASE WHEN t.sla_breach=1   THEN 1.0 ELSE 0.0 END)*100, 1)       AS taux_breach,
        ROUND(AVG(t.resolution_heures), 1)                                           AS resolution_moy_h,
        ROUND(AVG(t.csat), 2)                                                        AS csat_reel,
        ROUND(AVG(t.csat) - a.csat_moyen, 2)                                        AS ecart_csat,
        ROUND(AVG(CAST(t.reopened AS FLOAT))*100, 1)                               AS taux_reouverture,
        ROUND(AVG(CASE WHEN t.statut='Escalade' THEN 1.0 ELSE 0.0 END)*100, 1)    AS taux_escalade,
        RANK() OVER (ORDER BY AVG(CASE WHEN t.sla_breach=1 THEN 1.0 ELSE 0.0 END) ASC)  AS rank_sla,
        RANK() OVER (ORDER BY AVG(t.csat) DESC NULLS LAST)                          AS rank_csat
    FROM tickets t
    JOIN agents a ON t.agent_id = a.agent_id
    GROUP BY a.nom, a.tier, a.bureau, a.csat_moyen
    ORDER BY taux_breach ASC


In [ ]:
print("Performance agents (triee par taux breach croissant - meilleurs en premier) :")
display(df_agt)

top = df_agt.loc[df_agt["taux_breach"].idxmin(), "nom"]
bas = df_agt.loc[df_agt["taux_breach"].idxmax(), "nom"]
print(f"\nTop performer SLA : {top}")
print(f"Agent priorite coaching : {bas}")
print(f"Ecart : {df_agt['taux_breach'].max() - df_agt['taux_breach'].min():.1f} points de breach")


**Jean-Marc Dubois — Tier 3, Paris.** Rank SLA : 1, Rank CSAT : 1.
41.3% de breach, CSAT réel 4.69. Le meilleur agent toutes métriques confondues.

**Ramatou Diaby — Tier 1, Ouagadougou.** Rank SLA : 12, Rank CSAT : 11.
56.2% de breach, CSAT réel 3.50. Le profil qui nécessite le plus d'attention.

Le pattern est clair : **le tier corrèle directement avec la performance**.
Tous les Tier 3 sont dans le top 2. Tous les Tier 1 sont dans le bas du classement.

Mais regardons **l'écart CSAT** — il est négatif pour tous les agents.
Ça veut dire que sur la période analysée, tous les agents performent moins bien
que leur historique. Signal possible d'une dégradation générale de la qualité du service.

### Visualisation - Matrice performance 3-en-1

> **POINT PEDAGOGIQUE - La normalisation directionnelle dans la heatmap.**
>
> Le defi : `taux_breach` est mauvais quand eleve, mais `csat_reel` est bon quand eleve.
> Si on normalise les deux colonnes de la meme facon, rouge et vert ont des significations
> inversees selon la colonne - la heatmap devient impossible a lire.
>
> **La solution : normalisation directionnelle.**
>
> Pour `taux_breach` et `taux_reouverture` (mauvais quand eleve) :
> ```python
> norm = (valeur - min) / (max - min)   # 0 = meilleur, 1 = pire
> ```
>
> Pour `csat_reel` (bon quand eleve) - on inverse :
> ```python
> norm = 1 - (valeur - min) / (max - min)   # 0 = meilleur (CSAT eleve), 1 = pire
> ```
>
> Resultat : rouge = probleme et vert = bon sur **toutes** les colonnes simultanement.
> On annote avec les valeurs reelles (`annot=hm_data.round(1)`) pour garder la lisibilite.
>
> **Les quadrants du scatter breach vs CSAT :**
> - Gauche-haut (faible breach + CSAT eleve) : Top Performers
> - Droite-bas (fort breach + CSAT bas) : Priorite coaching immediat

In [ ]:
tier_colors = {"Tier 1": COLORS["warning"], "Tier 2": COLORS["primary"], "Tier 3": COLORS["secondary"]}
sc_c = [tier_colors.get(t, COLORS["neutral"]) for t in df_agt["tier"]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Scatter Breach vs CSAT
axes[0].scatter(df_agt["taux_breach"], df_agt["csat_reel"],
                s=df_agt["nb_tickets"] * 1.5 + 40, c=sc_c, alpha=0.85,
                edgecolors="white", linewidths=1.5)
for _, r in df_agt.iterrows():
    axes[0].annotate(r["nom"].split()[0], (r["taux_breach"], r["csat_reel"]),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[0].axvline(df_agt["taux_breach"].mean(), color="gray", linestyle="--", alpha=0.5)
axes[0].axhline(df_agt["csat_reel"].mean(),   color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Taux SLA breach (%)")
axes[0].set_ylabel("CSAT moyen")
axes[0].set_title("Matrice : Breach vs CSAT\n(taille=charge, couleur=tier)", fontweight="bold")
axes[0].legend(handles=[Patch(color=v, label=k) for k, v in tier_colors.items()], fontsize=9)

# Charge par agent
axes[1].barh(df_agt["nom"], df_agt["nb_tickets"], color=sc_c, alpha=0.85, edgecolor="white")
moy_c = df_agt["nb_tickets"].mean()
axes[1].axvline(moy_c, color=COLORS["neutral"], linestyle="--", linewidth=1.5, label=f"Moy. {moy_c:.0f}")
axes[1].set_title("Charge par agent (tickets traites)", fontweight="bold")
axes[1].set_xlabel("Nb tickets")
axes[1].legend(fontsize=9)

# Heatmap normalisation directionnelle
hm_data = df_agt[["nom","taux_breach","csat_reel","taux_reouverture"]].set_index("nom")
hm_norm = pd.DataFrame(index=hm_data.index)
for col in ["taux_breach", "taux_reouverture"]:
    r = hm_data[col].max() - hm_data[col].min()
    hm_norm[col] = (hm_data[col] - hm_data[col].min()) / r if r > 0 else 0.5
r_c = hm_data["csat_reel"].max() - hm_data["csat_reel"].min()
hm_norm["csat_reel"] = 1 - (hm_data["csat_reel"] - hm_data["csat_reel"].min()) / r_c if r_c > 0 else 0.5
hm_norm  = hm_norm[["taux_breach","csat_reel","taux_reouverture"]]
hm_data2 = hm_data[["taux_breach","csat_reel","taux_reouverture"]]

sns.heatmap(hm_norm, annot=hm_data2.round(1), fmt=".1f", cmap="RdYlGn_r",
             ax=axes[2], linewidths=0.5, linecolor="white",
             cbar_kws={"label": "Normalise (0=bon, 1=pire)"})
axes[2].set_title("Heatmap agents\n(rouge=probleme, vert=bon partout)", fontweight="bold")

plt.tight_layout()
plt.savefig("performance_agents.png", dpi=150, bbox_inches="tight")
plt.show()
print("Lecture : agents en vert sur toutes les colonnes = Top Performers")
print("          agents en rouge sur toutes les colonnes = Coaching immediat")

On a trois graphiques ici, et ils se lisent ensemble.

Chacun apporte une dimension que les autres n'ont pas.

**Le premier — la matrice Breach vs CSAT.**
>
>*Axe X* : taux SLA breach. *Axe Y* : CSAT moyen. *Taille des bulles* : charge de tickets. *Couleur* : tier.
>
>Les deux lignes pointillées gises découpent le graphique en 4 quadrants.
>
>La verticale à 47% — moyenne SLA breach.
>
>L'horizontale à 4.1 — moyenne CSAT.
>
>Le message est immédiat : les bulles se regroupent par couleur du haut-gauche vers le bas-droite.
>
>Vert en haut, violet au milieu, orange en bas.
>
>Jean-Marc Dubois — 41% breach, CSAT 4.7. Meilleur profil toutes métriques.
>
>Moussa et Ramatou — 55-56% breach, CSAT sous 3.5. Priorité coaching.
>
>*Le tier prédit la performance. C'est une corrélation quasi parfaite sur 12 agents.*


**Le deuxième change complètement la lecture du premier.**
>
>C'est la charge par agent — le nombre de tickets traités.
>
>Et là, quelque chose de décisif apparaît.
>
>Ramatou et Moussa — les deux agents les moins performants sur le scatter — ont une charge identique aux meilleurs agents.
>
>Environ 1 265 à 1 275 tickets chacun.
>
>Jean-Marc Dubois — 1 231 tickets. Même ordre de grandeur.
>
>Ça veut dire que la différence de performance entre Moussa et Jean-Marc n'est pas liée à la surcharge.
>
>À charge égale, l'écart de 15 points de breach vient d'ailleurs — compétences, méthode, expérience.
>
>C'est un argument décisif pour M. Diallo : la réponse n'est pas de réduire la charge des Tier 1.
>
>La réponse, c'est la montée en compétences.


**Le troisième — la heatmap** — c'est le tableau de bord complet de chaque agent sur 3 dimensions simultanément.

>**Colonne taux_breach.**
>
>Vert foncé en haut pour Jean-Marc à 41.3%.
>
>Rouge foncé en bas pour Ramatou à 56.2%.
>
>La gradation est parfaite — chaque ligne descend d'un cran.
>
>**Colonne csat_reel.**
>
>Même logique — vert en haut, rouge en bas.
>
>Jean-Marc à 4.7, Moussa à 3.4.
>
>On confirme visuellement ce que le scatter montrait déjà.
>
>**Colonne taux_reouverture.**
>
>Et c'est là que la heatmap révèle quelque chose d'inattendu.
>
>Ramatou Diaby — 56.2% breach, 3.5 CSAT — mais seulement 6.2% de réouverture. Le plus bas de tous.
>
>Moussa Kone — 55.3% breach, 3.4 CSAT — 6.9% de réouverture. Également parmi les plus bas.
>
>En revanche Samuel Acheampong — Tier 2, performances intermédiaires — 8.6% de réouverture. Le plus élevé.
>
>C'est contre-intuitif.
>
>Les agents qui breachent le plus ont paradoxalement moins de tickets réouverts.
>
>L'explication probable : leurs tickets mettent tellement de temps à être résolus que le client abandonne avant de relancer.
>Chez les bons agents, la résolution est rapide — alors le client revient si le problème réapparaît.
>
>Ce type d'insight, le scatter seul ne te le donne pas.
>
>Il faut la heatmap.

La synthèse en une phrase pour M. KOUAME :

*"Les agents Tier 1 portent la même charge que vos meilleurs agents, avec 15 points de breach en plus et un CSAT un point en dessous. Et leurs clients n'appellent plus — non pas parce qu'ils sont satisfaits, mais parce qu'ils ont renoncé."*

C'est ça que ces trois graphiques disent ensemble.

---

# 5. Evolution temporelle - LAG et heatmap heure x jour

> **POINT PEDAGOGIQUE - LAG() OVER : comparer a la periode precedente.**
>
> ```sql
> LAG(breach_pct) OVER (ORDER BY mois) AS breach_prev
> ```
>
> `LAG(colonne, n)` retourne la valeur de la colonne pour la ligne n positions avant
> dans la fenetre definie par `OVER(ORDER BY...)`. Par defaut n=1 - la ligne precedente.
>
> Pour la premiere ligne (premier mois), il n'existe pas de ligne precedente :
> LAG retourne NULL. C'est normal et attendu.
>
> **Sans LAG** : on voit le taux de breach de chaque mois isole.
> **Avec LAG** : on voit la variation - ce mois est-il meilleur ou pire que le precedent ?
>
> **La CTE (Common Table Expression) avec WITH :**
>
> ```sql
> WITH monthly AS (
>     SELECT DATE_TRUNC('month', ...) AS mois, COUNT(*), ...
>     FROM tickets GROUP BY 1
> )
> SELECT *, LAG(breach_pct) OVER (ORDER BY mois) FROM monthly
> ```
>
> La CTE calcule d'abord les agregats mensuels (1 ligne par mois).
> La requete principale applique ensuite LAG sur ces resultats agreges.
> On ne peut pas utiliser LAG directement dans un GROUP BY - la CTE resout ce probleme.
>
> **`GROUP BY 1` :** raccourci pour GROUP BY sur la premiere colonne du SELECT.
> Pratique mais a utiliser avec precaution - si l'ordre des colonnes change, le groupby aussi.
>
> **Heatmap volume heure x jour :**
> Repond a la question que les agregats classiques ne montrent pas :
> a quelles heures et quels jours les tickets arrivent-ils en masse ?
> `pivot_table(values="n", index="jour", columns="heure")` transforme les donnees
> du format long (une ligne par heure/jour) au format wide attendu par sns.heatmap.
> `fill_value=0` : si une combinaison n'existe pas, on met 0 plutot que NaN.

In [ ]:
%%sql df_evol <<
WITH monthly AS (
        SELECT
            DATE_TRUNC('month', CAST(created_at AS TIMESTAMP))               AS mois,
            COUNT(*)                                                           AS nb,
            ROUND(AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END)*100, 1)  AS breach_pct,
            ROUND(AVG(resolution_heures), 1)                                  AS res_moy,
            ROUND(AVG(csat), 2)                                               AS csat_moy
        FROM tickets
        GROUP BY 1
    )
    SELECT *,
           LAG(breach_pct) OVER (ORDER BY mois)                          AS breach_prev,
           ROUND(breach_pct - LAG(breach_pct) OVER (ORDER BY mois), 1)  AS delta_breach
    FROM monthly
    ORDER BY mois


In [ ]:
display(df_evol)

mois_amelio = (df_evol["delta_breach"] < 0).sum()
mois_degrad = (df_evol["delta_breach"] > 0).sum()
print(f"Mois d'amelioration : {mois_amelio} | Mois de degradation : {mois_degrad}")
print("Stable - pas d'amelioration structurelle" if abs(mois_amelio-mois_degrad) < 3 else "Tendance detectee")

Le premier mois — janvier 2022 — a `breach_prev = NULL` et `delta_breach = NULL`.
Normal : il n'y a pas de mois précédent.

Sur 3 ans de données, le taux de breach oscille entre 44% et 56%.
Pas de tendance baissière marquée. Le problème persiste.

Les mois de novembre-décembre affichent régulièrement des pics de volume.
Probablement liés aux fêtes et aux campagnes commerciales e-commerce.
Ces périodes concentrent les tickets et dégradent les délais.

In [ ]:
%%sql df_hm <<
SELECT
        jour_semaine,
        HOUR(CAST(created_at AS TIMESTAMP)) AS heure,
        COUNT(*) AS n
    FROM tickets
    GROUP BY jour_semaine, heure
    ORDER BY jour_semaine, heure


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x_idx = range(len(df_evol))
axes[0].plot(x_idx, df_evol["breach_pct"],
              color=COLORS["danger"], linewidth=2.5, marker="o", markersize=5)
axes[0].fill_between(x_idx, df_evol["breach_pct"]-3, df_evol["breach_pct"]+3,
                     alpha=0.12, color=COLORS["danger"])
moy = df_evol["breach_pct"].mean()
axes[0].axhline(moy, color=COLORS["neutral"], linestyle="--", linewidth=1.5, label=f"Moyenne : {moy:.1f}%")
axes[0].set_xticks(range(0, len(df_evol), 3))
axes[0].set_xticklabels(df_evol["mois"].astype(str).iloc[::3], rotation=45, ha="right", fontsize=9)
axes[0].set_title("Evolution mensuelle du taux SLA breach", fontweight="bold")
axes[0].set_ylabel("Taux breach (%)")
axes[0].legend(fontsize=9)

pivot_hm = df_hm.pivot_table(values="n", index="jour_semaine", columns="heure", fill_value=0)
jours = ["Lun","Mar","Mer","Jeu","Ven","Sam","Dim"]
pivot_hm.index = jours[:len(pivot_hm)]
sns.heatmap(pivot_hm, cmap="Blues", ax=axes[1], linewidths=0.3,
             linecolor="#F0F0F0", cbar_kws={"label": "Nb de tickets"})
axes[1].set_title("Volume de tickets - Heure x Jour de la semaine", fontweight="bold")
axes[1].set_xlabel("Heure de creation")

plt.tight_layout()
plt.savefig("evolution_temporelle.png", dpi=150, bbox_inches="tight")
plt.show()

pic = df_hm.loc[df_hm["n"].idxmax()]
print(f"Pic d'activite : {jours[int(pic['jour_semaine'])]} a {int(pic['heure'])}h - {int(pic['n'])} tickets")

---

# 6. Angles morts du systeme d'alerte SLA

> **POINT PEDAGOGIQUE - "Angles morts" : tickets en breach sans alerte.**
>
> La table `sla_alerts` recense les tickets en breach. Mais tous n'ont pas
> ete notifies a un superviseur. Un ticket en breach non notifie est invisible
> - il ne sera traite que quand le client se plaindra.
>
> **Le double filtre :**
> ```sql
> WHERE sa.statut_notification = 'Non notifie'
>   AND sa.manager_alerte = false
> ```
> Ces deux conditions identifient les cas les plus critiques.
>
> **`ORDER BY t.priorite ASC, sa.depassement_heures DESC` :**
> Double tri : priorite la plus haute (1) en premier, puis depassement le plus
> grand a priorite egale. Le premier ticket affiche est le plus urgent.
>
> **`CASE WHEN` pour les tranches (binning en SQL) :**
> L'equivalent SQL de `pd.cut()`. L'ordre des conditions est critique :
> SQL evalue de haut en bas et s'arrete a la premiere condition vraie.
> Toujours ecrire les tranches dans l'ordre croissant des valeurs.
>
> **`SUM(COUNT(*)) OVER ()` dans la requete des tranches :**
> Window function qui calcule le total global de tous les COUNT(*) sans rompre le GROUP BY.
> Equivalent pandas : `df['n'] / df['n'].sum() * 100`.
> C'est le seul moyen propre de calculer un pourcentage du total dans une agregation SQL.

In [ ]:
%%sql df_angles_morts <<
SELECT
        t.ticket_id, t.statut, t.priorite, t.resolution_heures, t.sla_heures,
        ROUND(t.resolution_heures - t.sla_heures, 1)   AS depassement_calcule_h,
        sa.depassement_heures,
        sa.statut_notification,
        sa.manager_alerte
    FROM tickets t
    JOIN sla_alerts sa ON t.ticket_id = sa.ticket_id
    WHERE sa.statut_notification = 'Non notifie'
      AND sa.manager_alerte = false
    ORDER BY t.priorite ASC, sa.depassement_heures DESC
    LIMIT 20


In [ ]:
print(f"Tickets en breach non notifies (top 20) : {len(df_angles_morts)}")
if len(df_angles_morts) > 0:
    display(df_angles_morts)
    print("Recommandation : automatiser les alertes a 80% du SLA consomme.")
else:
    print("Tous les tickets en breach ont ete notifies.")

In [ ]:
%%sql df_tranches <<
SELECT
        CASE
            WHEN sa.depassement_heures <= 2  THEN '1 - 0-2h'
            WHEN sa.depassement_heures <= 8  THEN '2 - 2-8h'
            WHEN sa.depassement_heures <= 24 THEN '3 - 8-24h'
            WHEN sa.depassement_heures <= 72 THEN '4 - 1-3j'
            ELSE '5 - 3j+'
        END AS tranche,
        COUNT(*)                                                                 AS nb_tickets,
        ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1)                        AS pct_total,
        ROUND(AVG(t.csat), 2)                                                    AS csat_moy,
        ROUND(AVG(CASE WHEN t.statut='Escalade' THEN 1.0 ELSE 0.0 END)*100, 1) AS taux_escalade
    FROM sla_alerts sa
    JOIN tickets t ON sa.ticket_id = t.ticket_id
    GROUP BY tranche
    ORDER BY tranche


In [ ]:
display(df_tranches)

pct_grave = df_tranches[df_tranches["tranche"].isin(["4 - 1-3j","5 - 3j+"])]["pct_total"].sum()
print(f"\n{pct_grave:.1f}% des breaches durent plus d'1 jour - intervention structurelle necessaire.")

---

# 7. Satisfaction client - CSAT

> **POINT PEDAGOGIQUE - Le NPS simplifie.**
>
> ```sql
> pct_5 - pct_1_2 AS nps_simplifie
> ```
>
> - `pct_5` : % de notes 5/5 (clients enchantes = promoteurs)
> - `pct_1_2` : % de notes 1 ou 2/5 (clients frustres = detracteurs)
>
> Un NPS positif = plus de promoteurs que de detracteurs.
> Un NPS negatif = plus de clients frustres qu'enchantes.
>
> **`WHERE csat IS NOT NULL` :** filtre explicite avant de calculer les proportions.
> Sans ce filtre, COUNT(*) inclurait les lignes sans CSAT, sous-estimant les taux.
>
> **Pourquoi prefixer les tranches avec des chiffres :**
> On a prefixe les tranches (`'1 - < 4h'`, `'2 - 4-12h'`, ...) pour forcer l'ordre
> chronologique dans le ORDER BY.
> Sans ce prefixe, le tri alphabetique donnerait `< 4h, 1-3j, 12-24h, 3j+, 4-12h`
> - ordre incorrect qui produirait un graphique illisible.

In [ ]:
%%sql df_csat <<
SELECT
        ROUND(AVG(csat), 2)                                                AS note_moyenne,
        COUNT(*)                                                            AS nb_avis,
        ROUND(SUM(CASE WHEN csat=5  THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS pct_5_etoiles,
        ROUND(SUM(CASE WHEN csat<=2 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS pct_insatisfaits,
        ROUND(SUM(CASE WHEN csat=5  THEN 1 ELSE 0 END)*100.0/COUNT(*) -
              SUM(CASE WHEN csat<=2 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS nps_simplifie
    FROM tickets
    WHERE csat IS NOT NULL


In [ ]:
print("SATISFACTION GLOBALE - AfriCare Support")
print("=" * 50)
for col, val in df_csat.iloc[0].items():
    print(f"  {col:<25} : {val}")

nps = df_csat["nps_simplifie"].iloc[0]
print("  NPS positif : plus de promoteurs" if nps > 0 else "  NPS negatif : plus de detracteurs")

In [ ]:
%%sql df_delay_csat <<
SELECT
        CASE
            WHEN resolution_heures <= 4   THEN '1 - < 4h'
            WHEN resolution_heures <= 12  THEN '2 - 4-12h'
            WHEN resolution_heures <= 24  THEN '3 - 12-24h'
            WHEN resolution_heures <= 72  THEN '4 - 1-3j'
            ELSE '5 - 3j+'
        END AS tranche_delai,
        ROUND(AVG(csat), 2) AS csat_moy,
        COUNT(*)             AS nb_avis
    FROM tickets
    WHERE csat IS NOT NULL
    GROUP BY tranche_delai
    ORDER BY tranche_delai


In [ ]:
ratings_dist = conn.execute("""
    SELECT CAST(csat AS INT) AS rating, COUNT(*) AS cnt
    FROM tickets WHERE csat IS NOT NULL
    GROUP BY CAST(csat AS INT) ORDER BY rating
""").df()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels_clean = [t.split(" - ")[1] for t in df_delay_csat["tranche_delai"]]
axes[0].plot(labels_clean, df_delay_csat["csat_moy"],
              color=COLORS["danger"], marker="o", linewidth=2.5, markersize=8)
axes[0].fill_between(labels_clean, 1, df_delay_csat["csat_moy"], alpha=0.1, color=COLORS["danger"])
for i, (lbl, val) in enumerate(zip(labels_clean, df_delay_csat["csat_moy"])):
    axes[0].annotate(f"{val:.2f}", (i, val), xytext=(0, 8), textcoords="offset points",
                      ha="center", fontsize=9, fontweight="bold")
axes[0].set_ylim(1, 5.2)
axes[0].set_title("CSAT moyen selon le delai de resolution", fontweight="bold")
axes[0].set_ylabel("CSAT moyen / 5")

colors_r = [COLORS["danger"],"#F0997B",COLORS["warning"],"#97C459",COLORS["secondary"]]
axes[1].bar(ratings_dist["rating"].astype(str), ratings_dist["cnt"],
             color=colors_r[:len(ratings_dist)], alpha=0.85, edgecolor="white")
for bar, val in zip(axes[1].patches, ratings_dist["cnt"]):
    pct = val / ratings_dist["cnt"].sum() * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                  f"{pct:.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[1].set_title(f"Distribution CSAT (moy : {df_csat['note_moyenne'].values[0]:.2f}/5)", fontweight="bold")

plt.tight_layout()
plt.savefig("satisfaction_csat.png", dpi=150, bbox_inches="tight")
plt.show()

c_r = df_delay_csat[df_delay_csat["tranche_delai"]=="1 - < 4h"]["csat_moy"].values[0]
c_l = df_delay_csat[df_delay_csat["tranche_delai"]=="5 - 3j+"]["csat_moy"].values[0]
print(f"Impact delai : < 4h -> {c_r:.2f}/5 | > 3j -> {c_l:.2f}/5 | Ecart : {c_r-c_l:.2f} points")

On segmente les tickets en 5 tranches de délai.

Les tickets résolus en moins de 4 heures — CSAT le plus élevé.

Les tickets qui traînent plus de 3 jours — CSAT le plus bas.

C'est la **preuve statistique** que l'opérationnel impacte directement la satisfaction.

Chaque heure de délai supplémentaire érode le score.

Ce n'est pas une intuition — c'est dans les données.

Et cette courbe, tu la montres à M. KOAUME pour justifier l'investissement dans l'amélioration des processus.

*"Monsieur KOAUME, réduire le délai de résolution de 3 jours à moins de 24 heures
améliore mécaniquement le CSAT. Les données le prouvent."*

---

# 8. Synthese des insights pour M. Kouame

> **POINT PEDAGOGIQUE - La synthese est le livrable le plus important.**
>
> M. Kouame ne lira pas 200 lignes de SQL. Il lira cette page.
>
> **Format recommande pour chaque insight :**
> Chiffre precis -> Signification metier -> Recommandation concrete et chiffree
>
> Les recommandations doivent etre **actionnables** :
> pas "ameliorer le SLA" mais "renégocier le SLA Fraude de 60h a 90h".
> pas "former les agents" mais "plan de mentorat Tier 3 -> Tier 1 sur 30 jours".

In [ ]:
print("=" * 70)
print("  SYNTHESE ANALYTIQUE - AFRICAIRE SUPPORT")
print("=" * 70)

kpi_v = df_kpi.iloc[0]
cat_p = df_sla_cat.iloc[0]
agt_b = df_agt.loc[df_agt["taux_breach"].idxmax()]
agt_t = df_agt.loc[df_agt["taux_breach"].idxmin()]
c_r   = df_delay_csat[df_delay_csat["tranche_delai"]=="1 - < 4h"]["csat_moy"].values[0]
c_l   = df_delay_csat[df_delay_csat["tranche_delai"]=="5 - 3j+"]["csat_moy"].values[0]

print(f"\nINSIGHT 1 - SLA en crise structurelle")
print(f"  {kpi_v['taux_sla_breach_pct']:.1f}% de breach - stable sur 30 mois.")
print(f"  Pas d'amelioration organique. Intervention structurelle necessaire.")
print(f"  Recommandation : renégocier les SLA sur categories breach > 50%.")

print(f"\nINSIGHT 2 - Categorie la plus critique")
print(f"  '{cat_p['nom']}' : {cat_p['taux_breach']:.1f}% breach")
print(f"  Depassement moyen : {cat_p['depassement_moy_h']:.0f}h au-dela du SLA de {cat_p['sla_heures']:.0f}h.")

print(f"\nINSIGHT 3 - Ecart de performance agents")
print(f"  Ecart : {df_agt['taux_breach'].max()-df_agt['taux_breach'].min():.0f} points de breach")
print(f"  Top : {agt_t['nom']} ({agt_t['taux_breach']:.1f}% breach, CSAT {agt_t['csat_reel']:.2f})")
print(f"  A coacher : {agt_b['nom']} ({agt_b['taux_breach']:.1f}% breach, CSAT {agt_b['csat_reel']:.2f})")
print(f"  Recommandation : mentorat Top Performers -> agents en difficulte (30 jours).")

print(f"\nINSIGHT 4 - Patterns temporels")
print(f"  Consulter la heatmap heure x jour pour adapter les plannings.")

print(f"\nINSIGHT 5 - CSAT et delai")
print(f"  Resolution < 4h -> CSAT {c_r:.2f}/5 | Resolution > 3j -> CSAT {c_l:.2f}/5")
print(f"  Recommandation : 90% des tickets resolus en < 24h pour CSAT cible 4.35+.")

print(f"\n{'='*70}")
print("  PATTERNS SQL MAITRISES")
print("=" * 70)
patterns = [
    "AVG(CASE WHEN cond THEN 1.0 ELSE 0.0 END) - calcul de taux",
    "JOIN multi-tables avec alias - enrichissement sans ambiguite",
    "RANK() OVER (ORDER BY ...) - classement des agents",
    "RANK() ... DESC NULLS LAST - gestion des NULLs",
    "LAG(col) OVER (ORDER BY ...) - comparaison periode precedente",
    "WITH ... AS (CTE) - requete en deux etapes lisibles",
    "CASE WHEN dans GROUP BY - binning (tranches de delai)",
    "SUM(COUNT(*)) OVER () - total global en window function",
    "Normalisation directionnelle - heatmap rouge=probleme / vert=bon",
]
for p in patterns:
    print(f"  - {p}")

---

## Recapitulatif du Notebook 3

| Section | Technique SQL principale | Insight produit |
|---|---|---|
| KPIs globaux | `AVG(CASE WHEN)` - `ROUND` | 47.3% de breach - objectif 10% |
| SLA par categorie | `JOIN` - `CASE WHEN` agrege | Top categories critiques |
| Performance agents | `RANK() OVER` - `NULLS LAST` | Ecart 30+ pts entre agents |
| Evolution temporelle | `WITH` CTE - `LAG() OVER` | Pas d'amelioration structurelle |
| Angles morts | `JOIN` sla_alerts - double filtre | Tickets breach non notifies |
| Tranches breach | `CASE WHEN` binning - `SUM() OVER()` | Gravite des depassements |
| CSAT et delai | NPS simplifie - filtre `NULL` | -1.6 pts CSAT entre < 4h et > 3j |

---

**Prochaine etape : Notebook 4 - Dashboard Power BI, DAX & Storytelling**

---

*DataProjectLab - apprendre la data sur des cas concrets, structures et orientes metier.*